# 7j — Weekly age-pair mean & excess degree

Longitudinal companion to `6j` §8. For every **age pair**
*(participant CIS bin → contactee CIS bin)*, tracked **per inc2prev-aligned week**
and split by **home** / **non-home**, we summarise the duration-weighted degree by

- its **mean** `m`, and
- its **excess degree** `m(1+CV²) = ⟨k²⟩/⟨k⟩` — the mean of the size-biased
  ("excess") degree, i.e. the expected weighted degree reached by following a random
  contact.

Both are **empirical** weekly moments (no per-cell distribution fit, which would be
unstable on small weekly cells). Statistics are **zero-excluded / positives-only per
cell**: a participant-day enters cell *i→j* only if it had ≥1 contact to bin *j*.

**Age grid** = the `age_school` rows of `inc2prev/data-processed/populations.csv`
(England): `2-10, 11-15, 16-24, 25-34, 35-49, 50-69, 70+`. Ages **< 2 are ignored**
(no population bin below 2). Where an age is reported only as an interval spanning
several bins (contactee "0–17", missing / "Don't know", …), the bin is drawn by
**weighted random sampling ∝ bin population** (single seed `MersenneTwister(1236)`),
exactly as in 6j §8.

We also visualise the **number of sampled participant-days** per age pair per week.

In [ ]:
include("main_utils.jl")
include("data_setup.jl")
include("comix_uk_time_series.jl")
include("vis_utils.jl")

using CSV, Random, StatsBase, Dates

default_plot_setting()

## §1 Load contacts and assign inc2prev-aligned weeks

Full CoMix-UK span (2020-03-23 → 2021-06-23), **not** date-filtered. Weeks are 7-day,
Sunday-start, anchored to `inc2prev_week_anchor()` (2021-03-21) so labels line up with
the inc2prev weekly series; each week is tagged by its **mid-date** (Wednesday).

In [ ]:
DMAX = 240   # >4h ⇒ weight 1 (same weighting scale as 6j)

df, df_part = read_comix_uk_contact_raw()   # full span, no date filter
println("participant-diary-days: ", nrow(unique(@select(df_part, :part_id_d, :date))))
println("date span: ", minimum(df_part.date), " → ", maximum(df_part.date))

# inc2prev-aligned 7-day week (Sunday-start); tag each date by its week mid-date.
WK_ANCHOR      = inc2prev_week_anchor()                 # 2021-03-21
week_index(d)  = fld((d - WK_ANCHOR).value, 7)
week_mid_of(d) = WK_ANCHOR + Day(week_index(d) * 7 + 3) # Wednesday of the week

println("weeks covered: ", length(unique(week_mid_of.(df_part.date))))

## §2 Age grid & population-weighted bin assignment

CIS bins from the `age_school` population table (England, `lo ≥ 2`), and a
population-weighted draw for any age interval overlapping more than one bin. Intervals
lying entirely below age 2 are dropped (`assign_bin → nothing`).

In [ ]:
# CIS bins + population weights (England, `age_school` rows of populations.csv).
pop_df  = CSV.read("../inc2prev/data-processed/populations.csv", DataFrame)
pop_age = @subset(pop_df, :level .== "age_school", :geography .== "England")
_asint(x)   = x isa AbstractString ? parse(Int, x)     : Int(x)
_asfloat(x) = x isa AbstractString ? parse(Float64, x) : Float64(x)
pop_age = @transform(pop_age, :lo = _asint.(:lower_age_limit), :pop = _asfloat.(:population))
pop_age = @subset(pop_age, :lo .>= 2)      # ignore any population bin below age 2
sort!(pop_age, :lo)

CIS_LO  = pop_age.lo                        # [2,11,16,25,35,50,70]
CIS_HI  = vcat(CIS_LO[2:end] .- 1, 120)     # [10,15,24,34,49,69,120]
CIS_POP = pop_age.pop
N_BIN   = length(CIS_LO)
CIS_LAB = [CIS_LO[j] == CIS_LO[end] ? "$(CIS_LO[j])+" : "$(CIS_LO[j])-$(CIS_HI[j])" for j in 1:N_BIN]
AGE_MIN = CIS_LO[1]                         # 2 — ages below this are ignored

_toint(s) = (ismissing(s) || s == "NA") ? nothing : tryparse(Int, String(s))

"Parse a participant age-group string \"lo-hi\" → (lo,hi); NA/unparseable → (0,120)."
function parse_age_interval(s)
    (ismissing(s) || s == "NA") && return (0, 120)
    parts = split(String(s), "-")
    length(parts) == 2 || return (0, 120)
    a = tryparse(Int, parts[1]); b = tryparse(Int, parts[2])
    (a === nothing || b === nothing) ? (0, 120) : (a, b)
end

"Contactee interval from (est_min, est_max); NA/unparseable → (0,120)."
function interval_from_minmax(mn, mx)
    a = _toint(mn); b = _toint(mx)
    (a === nothing || b === nothing) ? (0, 120) : (a, b)
end

overlapping(a, b) = [j for j in 1:N_BIN if a <= CIS_HI[j] && b >= CIS_LO[j]]

"CIS bin for [a,b]: `nothing` if entirely < 2 (ignored); deterministic if one
overlapping bin; else sampled ∝ population."
function assign_bin(a, b, rng)
    b < AGE_MIN && return nothing               # entirely below age 2 → ignore
    cand = overlapping(a, b)
    isempty(cand)     && return 1               # spans up to just under 2 → youngest bin
    length(cand) == 1 && return cand[1]
    return sample(rng, cand, Weights(CIS_POP[cand]))
end

is_ambiguous(a, b) = length(overlapping(a, b)) > 1

DataFrame(bin = CIS_LAB, lo = CIS_LO, hi = CIS_HI, population_M = round.(CIS_POP ./ 1e6; digits = 2))

## §3 Build the age-paired, week-tagged contact table

Re-read the arrow for the contactee-age columns and duration/`cnt_home` (as 6j §8),
draw a participant bin per participant-day and a contactee bin per contact (dropping
`nothing` = ages < 2), and tag each contact with its participant-day's week mid-date.

In [ ]:
craw = read_arrow_df("../dt_comix_no_public/contacts_uk.arrow";
    cols = [:part_wave_uid, :date, :cnt_home, :cnt_minutes_max, :cnt_total_time,
            :cnt_age_est_min, :cnt_age_est_max])
craw = @select(craw,
    :part_id_d      = :part_wave_uid,
    :date,
    :cnt_home,
    :duration_multi = _uk_duration_multi.(:cnt_minutes_max, :cnt_total_time),
    :cnt_age_est_min, :cnt_age_est_max)
standardise_cnt_home_values!(craw)

rng = MersenneTwister(1236)

# participant CIS bin: one draw per participant-day (drop <2)
piv = parse_age_interval.(df_part.part_age)
df_part[!, :part_bin] = [assign_bin(a, b, rng) for (a, b) in piv]
part_lookup = @subset(unique(@select(df_part, :part_id_d, :date, :part_bin)),
                      .!isnothing.(:part_bin))
part_lookup[!, :part_bin] = Int.(part_lookup.part_bin)

dfA = innerjoin(craw, part_lookup, on = [:part_id_d, :date])

# contactee CIS bin: one draw per contact (drop <2)
civ = [interval_from_minmax(mn, mx) for (mn, mx) in zip(dfA.cnt_age_est_min, dfA.cnt_age_est_max)]
dfA[!, :cnt_bin] = [assign_bin(a, b, rng) for (a, b) in civ]
dfA = @subset(dfA, .!isnothing.(:cnt_bin))
dfA[!, :cnt_bin] = Int.(dfA.cnt_bin)

# tag each contact with its participant-day's inc2prev week mid-date
dfA[!, :week_mid] = week_mid_of.(dfA.date)

println("contacts retained: ", nrow(dfA))
println("contactee ages ambiguous (span >1 CIS bin): ",
        round(100 * mean(is_ambiguous.(first.(civ), last.(civ))); digits = 1), "%")
println("weeks: ", length(unique(dfA.week_mid)))
first(dfA, 4)

## §4 Weekly per-cell mean, excess degree, and sample count

For each **week × (participant bin → contactee bin) × setting**, the per-participant-day
weighted degree (sum of `duration_weight` over that day's contacts into the cell —
positive by construction, i.e. **zero-excluded**) gives the sample size `n`, mean `m`,
CV, and excess degree `m(1+CV²)`.

In [ ]:
function week_cell_stats(setting)
    sub = setting === :home ? @subset(dfA, :cnt_home .== "true") :
                              @subset(dfA, :cnt_home .== "false")
    sub = @transform(sub, :w = duration_weight.(:duration_multi, DMAX))
    # per participant-day × week × cell: summed weighted degree (positive)
    g = combine(groupby(sub, [:part_id_d, :date, :week_mid, :part_bin, :cnt_bin]),
                :w => sum => :deg)
    # per week × cell: n, mean, CV, excess
    stats = combine(groupby(g, [:week_mid, :part_bin, :cnt_bin]),
        :deg => length => :n,
        :deg => mean   => :mean,
        :deg => (x -> length(x) > 1 ? std(x) / mean(x) : 0.0) => :cv)
    @transform!(stats, :excess = :mean .* (1 .+ :cv .^ 2), :setting = string(setting))
    return stats
end

statsall = vcat(week_cell_stats(:home), week_cell_stats(:nonhome))
sort!(statsall, [:setting, :part_bin, :cnt_bin, :week_mid])
@assert all(statsall.excess .>= statsall.mean .- 1e-9)   # excess ≥ mean always
println("rows (week×cell×setting): ", nrow(statsall))
statsall

## §5 Mean & excess degree over weeks, per age pair

One 7×7 grid per setting (row = participant bin, col = contactee bin). Each panel plots
the weekly **mean** (solid black) and **excess degree** m(1+CV²) (solid red); the gap
is the tail-driven excess. Both are drawn as solid lines so short/sparse segments stay
visible. **Every week with ≥ 1 sampled participant-day is drawn** (no sample-size
threshold), so coverage matches the §6 count grid — read those counts alongside these
series, since low-`n` weeks (common in the 70+ cells) give noisier mean/excess estimates.
Axes are shared for comparability.

In [ ]:
mkpath("../res")

WEEKS   = sort(unique(statsall.week_mid))
WK_XLIM = (WEEKS[1], WEEKS[end])
_tickpos = WEEKS[round.(Int, range(1, length(WEEKS); length = 4))]
WK_TICKS = (_tickpos, Dates.format.(_tickpos, "u-yy"))

# index: (setting, i, j) → weekly stats sorted by week (all weeks with ≥1 sample)
statidx = Dict{Tuple{String,Int,Int},DataFrame}()
for gp in groupby(statsall, [:setting, :part_bin, :cnt_bin])
    statidx[(gp.setting[1], gp.part_bin[1], gp.cnt_bin[1])] = sort(DataFrame(gp), :week_mid)
end

# per-setting shared y-limit: comparable across a grid's 49 panels, scaled to that
# setting's own range (home is far lighter-tailed than non-home).
setting_ymax(s) = 1.05 * maximum(k -> k[1] == string(s) ? maximum(statidx[k].excess) : 0.0,
                                 keys(statidx))

function me_panel(s, i, j, ylim)
    p = plot(; legend = false, tickfontsize = 4, guidefontsize = 5,
             xlim = WK_XLIM, ylim = ylim, xticks = WK_TICKS, xrotation = 45)
    d = get(statidx, (string(s), i, j), nothing)
    if d !== nothing
        plot!(p, d.week_mid, d.mean;   color = :black, lw = 1.0,
              marker = :circle, markersize = 1.2, markerstrokewidth = 0)
        plot!(p, d.week_mid, d.excess; color = :red, lw = 1.0,
              marker = :circle, markersize = 1.2, markerstrokewidth = 0)
    end
    annotate!(p, WEEKS[end], 0.96 * ylim[2],
              text("$(CIS_LAB[i])→$(CIS_LAB[j])", 5, :right, :top))
    p
end

function me_grid(s)
    ylim = (0.0, setting_ymax(s))
    panels = [me_panel(s, i, j, ylim) for i in 1:N_BIN for j in 1:N_BIN]
    p = plot(panels...; layout = (N_BIN, N_BIN), size = (1700, 1500),
             plot_title = "Weekly mean (black) & excess degree m(1+CV^2) (red) — $(s)  " *
                          "(row = participant bin, col = contactee bin)",
             plot_titlefontsize = 11)
    savefig(p, "../res/7j_agepair_mean_excess_weekly_$(s).png")
    p
end
nothing

In [ ]:
me_grid(:home)

In [ ]:
me_grid(:nonhome)

## §6 Sample counts per age pair over weeks

Number of sampled participant-days `n` per age pair per week (log y-axis), home (blue)
vs non-home (red). Shows which cells are well-sampled enough for the mean/excess series
above to be reliable.

**`n` is zero-excluded** (same construction as §4): it counts only participant-days with
**≥ 1** contact into that (participant bin → contactee bin) cell that week — *not* every
participant surveyed. A participant-day reporting zero contacts into a cell produces no
row for it (`dfA` is a contact-level table), so it contributes to neither `n` nor the
mean/excess above. These are therefore statistics **conditional on contact**; a
zero-included denominator (participants at risk) would require joining against the full
participant-day roster, which this notebook does not do.

In [ ]:
cntidx = Dict{Tuple{String,Int,Int},DataFrame}()
for gp in groupby(statsall, [:setting, :part_bin, :cnt_bin])
    cntidx[(gp.setting[1], gp.part_bin[1], gp.cnt_bin[1])] = sort(DataFrame(gp), :week_mid)
end
CNT_YMAX = maximum(statsall.n)
CNT_YLIM = (1.0, 10.0^ceil(log10(CNT_YMAX)))

function cnt_panel(i, j)
    p = plot(; legend = false, tickfontsize = 4, guidefontsize = 5,
             yscale = :log10, xlim = WK_XLIM, ylim = CNT_YLIM,
             xticks = WK_TICKS, xrotation = 45)
    for (s, col) in (("home", :steelblue), ("nonhome", :crimson))
        d = get(cntidx, (s, i, j), nothing)
        d === nothing && continue
        plot!(p, d.week_mid, d.n; color = col, lw = 1.0,
              marker = :circle, markersize = 1.2, markerstrokewidth = 0)
    end
    annotate!(p, WEEKS[end], 10.0^(0.96 * log10(CNT_YLIM[2])),
              text("$(CIS_LAB[i])→$(CIS_LAB[j])", 5, :right, :top))
    p
end

cnt_fig = plot([cnt_panel(i, j) for i in 1:N_BIN for j in 1:N_BIN]...;
    layout = (N_BIN, N_BIN), size = (1700, 1500),
    plot_title = "Weekly sample count n  (home = blue, non-home = red)  " *
                 "(row = participant bin, col = contactee bin)",
    plot_titlefontsize = 11)
savefig(cnt_fig, "../res/7j_agepair_counts_weekly.png")
cnt_fig

## §7 Notes

- **Mean vs excess gap**: where the red (excess) sits far above the black (mean), the
  weekly weighted-degree distribution in that cell is over-dispersed (heavy right tail)
  — following a random contact reaches a much higher-degree day than the average.
  Non-home cells show the widest gaps.
- Mean & excess are **empirical**, **zero-excluded** weekly moments (positives-only per
  cell). Every week with ≥ 1 sampled participant-day is plotted (no sample-size
  threshold), so the mean/excess grids share the §6 count grid's coverage; weeks with
  small `n` (common in the sparse 70+ / off-diagonal cells) are noisier — cross-check
  against §6.
- Age bins are the `age_school` population grid (≥ 2). Ambiguous / missing ages are
  assigned by a single seeded population-weighted draw (`MersenneTwister(1236)`);
  re-running with another seed perturbs sparse off-diagonal cells most.
- Weeks are inc2prev-aligned (Sunday-start, anchor 2021-03-21), labelled by mid-date;
  early-2020 weeks are sparse for many age pairs.